In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 모듈 자동 리로드 설정
%load_ext autoreload
%autoreload 2

# src 폴더 경로 설정
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

# 공통 함수 모듈 임포트
from src import common_utils as utils
from src import config

utils.log("✅ 환경 설정 완료")

[18:52:02] ✅ [Config] 환경 설정 완료 (OS: Windows)
[18:52:02] ✅ 환경 설정 완료


In [2]:
# 1. 파일 경로
FILE_PATH = "../../data/10_processed/df_sha.parquet"

# 2. 학습/검증 기간 설정 (2~10월 Train, 11~12월 Test)
# 실제 데이터의 월 형식('202302' 등)에 맞춰 문자열 리스트 생성
TRAIN_MONTHS = [str(202300 + i) for i in range(2, 11)] # ['202302', ..., '202310']
TEST_MONTHS = ['202311', '202312']

print(f"📌 학습 기간(Train): {TRAIN_MONTHS}")
print(f"📌 검증 기간(Test): {TEST_MONTHS}")

📌 학습 기간(Train): ['202302', '202303', '202304', '202305', '202306', '202307', '202308', '202309', '202310']
📌 검증 기간(Test): ['202311', '202312']


In [3]:
# ==========================================================
# 데이터 전처리 및 안전한 결측치 제거
# ==========================================================
df = utils.load_parquet(FILE_PATH)

# 1. 전처리 수행
df = utils.data_preprocess(df)

# 2. 삭제 전 데이터 확인
initial_len = len(df)
null_rows = df.isnull().any(axis=1).sum()

print(f"📉 전체 {initial_len:,}건 중 결측치 포함 행: {null_rows:,}건 ({(null_rows/initial_len)*100:.1f}%)")

# 3. [안전장치] 만약 데이터가 너무 많이 삭제된다면 경고
if null_rows > initial_len * 0.5: # 50% 이상 날아가면 중단
    print("🚨 [경고] 데이터의 50% 이상이 삭제될 예정입니다! config.py의 mapping 규칙을 다시 확인하세요.")
    # 어떤 컬럼이 문제인지 출력
    print(df.isnull().sum().sort_values(ascending=False).head(5))
else:
    # 4. 결측치 제거
    df = df.dropna()
    print(f"✅ 전처리 완료: {len(df):,}건 (삭제됨: {initial_len - len(df):,}건)")
    
    # 결과 확인
    utils.i_view(df)

[18:52:22] 📂 로드 완료: 22,473,534 Rows (Selected 39 Columns)
[18:52:22] 🚀 데이터 전처리(data_preprocess) 시작...
[18:53:12] ✨ 전처리 완료: 37개 컬럼 변환 및 정리됨


📉 전체 22,473,534건 중 결측치 포함 행: 3,109,562건 (13.8%)
✅ 전처리 완료: 19,363,972건 (삭제됨: 3,109,562건)

📊 [Structure] Shape: (19363972, 39)


,SHA2_HASH,DERIVED_SVC_USE_DAYS_GRP,DERIVED_MEDIA_NM_GRP,DERIVED_PROD_NM_GRP,DERIVED_PROD_OLD_YN,DERIVED_PROD_ONE_PLUS_YN,DERIVED_AGMT_KIND_NM,DERIVED_STB_RES_1M_YN,DERIVED_SVOD_SCRB_CNT_GRP,DERIVED_PAID_CHNL_CNT_GRP,...,DERIVED_SMS_SEND_CLS_NM,DERIVED_CH_HH_AVG_MONTH1,DERIVED_CH_25_RATIO_MONTH1,DERIVED_CH_25_RATIO_MEAN_3MM,DERIVED_CH_FAV_RNK1,DERIVED_KIDS_USE_PV_MONTH1,DERIVED_NFX_USE_YN,DERIVED_YTB_USE_YN,P_MT,derived_cancel_yn
0,0000113b86db7c509bbe74d609529031b03b7c033dbdfb...,nan,HD,nan,0,1,신규,0,0.0,0.0,...,1,0.00,0.00,0.00,기타,0.0,0,0,202302,0
1,0000113b86db7c509bbe74d609529031b03b7c033dbdfb...,nan,HD,nan,0,1,신규,1,0.0,0.0,...,1,6.72,3.33,3.33,JTBC,0.0,0,0,202303,0
2,0000113b86db7c509bbe74d609529031b03b7c033dbdfb...,nan,HD,nan,0,1,신규,0,0.0,0.0,...,1,9.86,3.71,3.71,기타,0.0,0,0,202304,0
3,0000113b86db7c509bbe74d609529031b03b7c033dbdfb...,nan,HD,nan,0,1,신규,0,0.0,0.0,...,1,5.95,3.57,3.57,기타,0.0,0,0,202305,0
4,0000113b86db7c509bbe74d609529031b03b7c033dbdfb...,nan,HD,nan,0,1,신규,0,0.0,0.0,...,1,4.03,6.51,6.51,기타,0.0,0,0,202306,0



🔍 [Quality Summary]


,Dtype,Null_Count,N_Unique
SHA2_HASH,object,0,1965332
DERIVED_SVC_USE_DAYS_GRP,object,0,1
DERIVED_MEDIA_NM_GRP,object,0,3
DERIVED_PROD_NM_GRP,object,0,3
DERIVED_PROD_OLD_YN,object,0,2
DERIVED_PROD_ONE_PLUS_YN,object,0,2
DERIVED_AGMT_KIND_NM,object,0,7
DERIVED_STB_RES_1M_YN,object,0,2
DERIVED_SVOD_SCRB_CNT_GRP,float64,0,4
DERIVED_PAID_CHNL_CNT_GRP,float64,0,3


In [4]:
# ==========================================================
# OOT(Out-of-Time) 분할 및 다운샘플링
# ==========================================================
# config.py에 정의된 타겟 컬럼명 가져오기 (예: 'derived_cancel_yn')
target_col_name = config.DEFAULT_COLUMN_RULES['cancel_yn'].get('new_name', 'derived_cancel_yn')

# utils.split_oot_dataset 함수 활용
# 1. 월별(Train/Test) 데이터 분리
# 2. Train 데이터에 한해 다운샘플링 적용 (sampling_ratio=3.0 -> 해지:유지 = 1:3 비율)
X_train, y_train, X_test, y_test = utils.split_oot_dataset(
    df, 
    train_months=TRAIN_MONTHS, 
    test_months=TEST_MONTHS,
    target_col=target_col_name,
    sampling_ratio=3.0  # 데이터가 너무 크면 이 비율을 1.0~2.0으로 낮추거나 None으로 설정
)

print(f"✅ 최종 학습 데이터(Train): {X_train.shape}")
print(f"✅ 최종 검증 데이터(Test):  {X_test.shape}")

[18:53:57] 🚀 데이터셋 분할(split_oot_dataset) 시작...
[18:54:11] 📊 분할 결과: Train 15,868,607건 (기간: 202302~202310), Test 3,495,365건
[18:54:23] ✂️ 다운샘플링 적용 (비율 1:3.0): Train 데이터가 3,676,120건으로 조정됨


✅ 최종 학습 데이터(Train): (3676120, 36)
✅ 최종 검증 데이터(Test):  (3495365, 36)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score # [중요] recall_score 추가
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd

# ==========================================================
# 1. [핵심] Light Mode: 데이터 샘플링 (속도 향상)
# ==========================================================
# 전체 데이터의 10%만 사용하여 빠르게 우열을 가립니다.
SAMPLE_RATIO = 0.1 

print(f"📉 [Light Mode] 데이터 샘플링 시작 (비율: {SAMPLE_RATIO*100}%)")

# Train 데이터 샘플링
X_train_small = X_train.sample(frac=SAMPLE_RATIO, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

# Test 데이터 샘플링 (검증 속도 향상)
X_test_small = X_test.sample(frac=SAMPLE_RATIO, random_state=42)
y_test_small = y_test.loc[X_test_small.index]

print(f" - Train: {len(X_train):,} -> {len(X_train_small):,}건")
print(f" - Test:  {len(X_test):,} -> {len(X_test_small):,}건")


# ==========================================================
# 2. 모델 정의 (경량화 설정)
# ==========================================================
models = {
    # n_jobs=-1: 모든 CPU 코어 사용
    "LogisticRegression": LogisticRegression(random_state=42, max_iter=500, n_jobs=-1),
    
    "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=10),
    
    # n_estimators를 50으로 줄여 속도 향상 (비교용으로 충분)
    "RandomForest": RandomForestClassifier(random_state=42, n_estimators=50, max_depth=10, n_jobs=-1),
    
    "LGBM": LGBMClassifier(random_state=42, n_estimators=50, max_depth=10, verbose=-1, n_jobs=-1),
    
    "XGBoost": XGBClassifier(random_state=42, n_estimators=50, max_depth=10, eval_metric='logloss', n_jobs=-1)
}


# ==========================================================
# 3. 전처리 파이프라인 구성
# ==========================================================
# 데이터 타입에 따라 자동으로 컬럼 분류 (샘플 데이터 기준)
numeric_features = X_train_small.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train_small.select_dtypes(include=['object', 'category']).columns

print(f"\n🔢 수치형 변수({len(numeric_features)}): {list(numeric_features)}")
print(f"🔤 범주형 변수({len(categorical_features)}): {list(categorical_features)}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])


# ==========================================================
# 4. 모델 학습 및 평가 루프 (Recall 포함)
# ==========================================================
results = []

print("\n🚀 [재현율(Recall) 기준] 알고리즘 비교 시작 (Light Mode)...")
print("=" * 60)

for name, model in models.items():
    print(f"Training {name}...", end=" ") # 줄바꿈 없이 출력
    
    # 파이프라인 생성 (전처리 -> 모델)
    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('classifier', model)])
    
    # [중요] 샘플링된 데이터(small)로 학습
    clf.fit(X_train_small, y_train_small)
    
    # [중요] 샘플링된 데이터(small)로 예측
    y_pred = clf.predict(X_test_small)
    
    # predict_proba 지원 여부 확인
    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(X_test_small)[:, 1]
        roc = roc_auc_score(y_test_small, y_prob)
    else:
        roc = 0.5 # 지원 안함
    
    # 성능 지표 계산
    acc = accuracy_score(y_test_small, y_pred)
    f1 = f1_score(y_test_small, y_pred)
    rec = recall_score(y_test_small, y_pred) # [추가] 재현율 계산
    
    print(f"-> 완료 (Recall: {rec:.4f})")
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1 Score": f1,
        "Recall": rec,   # [추가] 결과 저장
        "ROC AUC": roc
    })

print("=" * 60)
print("✅ 비교 완료!")


# ==========================================================
# 5. 결과 출력 (Recall 기준 정렬)
# ==========================================================
# [중요] Recall이 높은 순서대로 정렬
results_df = pd.DataFrame(results).sort_values(by="Recall", ascending=False)
display(results_df)

# Recall 1등 모델 선정
best_model_name = results_df.iloc[0]['Model']
best_recall = results_df.iloc[0]['Recall']
print(f"\n🏆 Best Recall Model: {best_model_name} (Recall: {best_recall:.4f})")

📉 [Light Mode] 데이터 샘플링 시작 (비율: 10.0%)
 - Train: 3,676,120 -> 367,612건
 - Test:  3,495,365 -> 349,536건

🔢 수치형 변수(14): ['DERIVED_SVOD_SCRB_CNT_GRP', 'DERIVED_PAID_CHNL_CNT_GRP', 'DERIVED_INHOME_RATE', 'DERIVED_TOTAL_USED_DAYS', 'DERIVED_TV_SCRB', 'DERIVED_ANALOG_SCRB', 'DERIVED_DIGITAL_SCRB', 'DERIVED_TOTAL_INTERNET_SCRB', 'DERIVED_GIGA_INTERNET_SCRB', 'DERIVED_TV_I_CNT', 'DERIVED_CH_HH_AVG_MONTH1', 'DERIVED_CH_25_RATIO_MONTH1', 'DERIVED_CH_25_RATIO_MEAN_3MM', 'DERIVED_KIDS_USE_PV_MONTH1']
🔤 범주형 변수(22): ['DERIVED_SVC_USE_DAYS_GRP', 'DERIVED_MEDIA_NM_GRP', 'DERIVED_PROD_NM_GRP', 'DERIVED_PROD_OLD_YN', 'DERIVED_PROD_ONE_PLUS_YN', 'DERIVED_AGMT_KIND_NM', 'DERIVED_STB_RES_1M_YN', 'DERIVED_SCRB_PATH_NM_GRP', 'DERIVED_AGMT_END_SEG', 'DERIVED_AGMT_END_YMD', 'DERIVED_BUNDLE_YN', 'DERIVED_DIGITAL_GIGA_YN', 'DERIVED_DIGITAL_ALOG_YN', 'DERIVED_CH_LAST_DAYS_BF_GRP', 'DERIVED_VOC_TOTAL_MONTH1_YN', 'DERIVED_VOC_STOP_CANCEL_MONTH1_YN', 'DERIVED_AGE_GRP10', 'DERIVED_EMAIL_RECV_CLS_NM', 'DERIVED_SMS_SEND

,Model,Accuracy,F1 Score,Recall,ROC AUC
4,XGBoost,0.943525,0.137464,0.165788,0.720883
0,LogisticRegression,0.944260,0.125578,0.147449,0.686358
1,DecisionTree,0.940965,0.115099,0.141442,0.691264
3,LGBM,0.950692,0.125704,0.130586,0.708058
2,RandomForest,0.972893,0.003366,0.001686,0.677428



🏆 Best Recall Model: XGBoost (Recall: 0.1658)
